In [1]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt

# Undistort Images

In [2]:
"""
Implement the number of vertical and horizontal corners
"""
nb_vertical = 6
nb_horizontal = 9

# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((nb_horizontal*nb_vertical,3), np.float32)
objp[:,:2] = np.mgrid[0:nb_vertical,0:nb_horizontal].T.reshape(-1,2)

# Arrays to store object points and image points from all the images.
left_objpoints = [] # 3d point in real world space
right_objpoints = []
left_imgpoints = [] # 2d points in image plane.
right_imgpoints = []
left_images = glob.glob('imgs/rs/left/*.png')
right_images = glob.glob('imgs/rs/right/*.png')

for fname in left_images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (nb_vertical,nb_horizontal), None)

    # If found, add object points, image points (after refining them)
    if ret == True:
        left_objpoints.append(objp)
        left_imgpoints.append(corners)

for fname in right_images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (nb_vertical,nb_horizontal), None)

    # If found, add object points, image points (after refining them)
    if ret == True:
        right_objpoints.append(objp)
        right_imgpoints.append(corners)

In [3]:
left_ret, left_mtx, left_dist, left_rvecs, left_tvecs = cv2.calibrateCamera(left_objpoints, left_imgpoints, gray.shape[::-1], None, None)
right_ret, right_mtx, right_dist, right_rvecs, right_tvecs = cv2.calibrateCamera(right_objpoints, right_imgpoints, gray.shape[::-1], None, None)
img = cv2.imread('imgs/rs/left/left-0004.png')
h,  w = img.shape[:2]
left_newcameramtx, left_roi = cv2.getOptimalNewCameraMatrix(left_mtx,left_dist,(w,h),1,(w,h))
right_newcameramtx, right_roi = cv2.getOptimalNewCameraMatrix(right_mtx,right_dist,(w,h),1,(w,h))

In [4]:
def undistort_image(img, mtx, dist, newcameramtx, roi):
    # undistort
    dst = cv2.undistort(img, mtx, dist, None, newcameramtx)

    # crop the image
    x,y,w,h = roi
    dst = dst[y:y+h, x:x+w]
    return dst

In [5]:
import glob
import os

# Store both filenames and images
left_image_data = []
right_image_data = []
for img_path in glob.glob("imgs/rs/left/*.png"):
    img = cv2.imread(img_path)
    left_image_data.append((img_path, img))

for img_path in glob.glob("imgs/rs/right/*.png"):
    img = cv2.imread(img_path)
    right_image_data.append((img_path, img))

for img_path, img in left_image_data:
    dst = undistort_image(img, left_mtx, left_dist, left_newcameramtx, left_roi)
    # Save the image using the original filename
    filename = os.path.basename(img_path)
    cv2.imwrite(f"imgs/rs/undistorted/left/{filename}", dst)

for img_path, img in right_image_data:
    dst = undistort_image(img, right_mtx, right_dist, right_newcameramtx, right_roi)
    # Save the image using the original filename
    filename = os.path.basename(img_path)
    cv2.imwrite(f"imgs/rs/undistorted/right/{filename}", dst)

# Rectify Images

In [6]:
# Stereo Calibration
retval, cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, R, T, E, F = cv2.stereoCalibrate(
    left_objpoints, left_imgpoints, right_imgpoints,
    left_mtx, left_dist, right_mtx, right_dist,  gray.shape[::-1], None, None, None, None, cv2.CALIB_FIX_INTRINSIC
)

# Stereo Rectification
R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, gray.shape[::-1], 
    R, T, None, None, None, None, None, cv2.CALIB_ZERO_DISPARITY
)

# Undistort and Rectify
mapx1, mapy1 = cv2.initUndistortRectifyMap(cameraMatrix1, distCoeffs1, R1, P1, gray.shape[::-1], cv2.CV_16SC2)
mapx2, mapy2 = cv2.initUndistortRectifyMap(cameraMatrix2, distCoeffs2, R2, P2, gray.shape[::-1], cv2.CV_16SC2)

In [7]:
# Store both filenames and images
left_image_data = []
right_image_data = []
for img_path in glob.glob("imgs/rs/undistorted/left/*.png"):
    img = cv2.imread(img_path)
    left_image_data.append((img_path, img))

for img_path in glob.glob("imgs/rs/undistorted/right/*.png"):
    img = cv2.imread(img_path)
    right_image_data.append((img_path, img))

for img_path, img in left_image_data:
    left_rectified = cv2.remap(img, mapx1, mapy1, cv2.INTER_LINEAR)
    filename = os.path.basename(img_path)
    cv2.imwrite(f"imgs/rs/rectified/left/{filename}", left_rectified)

for img_path, img in right_image_data:
    right_rectified = cv2.remap(img, mapx2, mapy2, cv2.INTER_LINEAR)
    filename = os.path.basename(img_path)
    cv2.imwrite(f"imgs/rs/rectified/right/{filename}", right_rectified)
